# GenBio-PathFM latent space
The current labless leader (0.6926) as a reference point for nanopath. Embed a random slice of the held-out (patient-split) val tiles with the frozen ViT-G, then look at the CLS geometry (PCA, spread, site batch effect), how patch tokens relate to CLS, and how redundant GenBio's three per-RGB-plane passes are. CLS spread numbers are directly comparable to `cls_analysis.ipynb` on our own checkpoints.

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import v2

from baselines.genbio_pathfm_baseline import load_probe_model
from dataloader import TCGATileDataset

In [ ]:
GENBIO = Path("/data/genbio-pathfm")
CONFIG = REPO / "configs/baseline.yaml"  # only its data split (dataset_dir, split_seed, val_fraction) is used
MEAN, STD = [0.697, 0.575, 0.728], [0.188, 0.240, 0.187]  # GenBio's config.json stats, same as the labless baseline run
N_SAMPLES, BATCH, SEED = 2048, 64, 0
device = torch.device("cuda")

## Model loading + val-set inference

In [ ]:
# Same adapter the frozen-probe baseline uses, so tokens match what the probes saw.
model = load_probe_model(GENBIO, device)

# Val split with deterministic preprocessing: the whole 224px tile, GenBio-normalized, one view, no local crops.
cfg = yaml.safe_load(os.path.expandvars(CONFIG.read_text()))
val_ds = TCGATileDataset({**cfg, "config_path": str(CONFIG)}, is_train=False)
val_ds.global_views, val_ds.local_views = 1, 0
val_ds.global_aug = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])
sample_idx = random.Random(SEED).sample(range(len(val_ds)), N_SAMPLES)
print(f"val tiles: {len(val_ds):,}  sampled: {N_SAMPLES}  slides in sample: {len(set(val_ds.slide_of[sample_idx]))}")

In [ ]:
# One forward pass over the sampled tiles. Full patch tokens would be ~5GB at 2048 tiles x 196 x 4608, so keep per-tile reductions instead:
# raw per-pass CLS, mean patch token, patch-CLS cosine maps, a few random patches per tile as a PCA background, and 32px thumbnails.
@torch.no_grad()
def embed(indices, bg_per_tile=8):
    out = {"cls": [], "patch_mean": [], "patch_cls_cos": [], "bg_patches": [], "thumb": []}
    gen = torch.Generator(device=device).manual_seed(SEED)
    mean, std = torch.tensor(MEAN, device=device)[:, None, None], torch.tensor(STD, device=device)[:, None, None]
    for batch in DataLoader(Subset(val_ds, indices), batch_size=BATCH, num_workers=8):
        x = batch["global_views"][:, 0].to(device, non_blocking=True)
        b = len(x)
        with torch.autocast(device.type, dtype=torch.bfloat16):
            cls, patches = model._encode(x.reshape(b * 3, 1, *x.shape[-2:]))  # GenBio is single-channel: each RGB plane is its own pass
        cls, patches = cls.float().unflatten(0, (b, 3)), patches.float().unflatten(0, (b, 3))  # (b, 3, D), (b, 3, P, D)
        # The probe embedding concatenates the three passes, so similarity on concatenated tokens is what probes see.
        cls_cat = F.normalize(cls.flatten(1), dim=-1)
        patch_cat = F.normalize(patches.transpose(1, 2).flatten(2), dim=-1)  # (b, P, 3D)
        per_pass = (F.normalize(patches, dim=-1) @ F.normalize(cls, dim=-1)[..., None])[..., 0]  # (b, 3, P)
        out["cls"].append(cls.cpu())
        out["patch_mean"].append(patch_cat.mean(1).cpu())
        out["patch_cls_cos"].append(torch.cat([(patch_cat @ cls_cat[..., None]).transpose(1, 2), per_pass], 1).cpu())  # (b, 4, P): concat, R, G, B
        keep = torch.rand(b, patch_cat.shape[1], device=device, generator=gen).argsort(1)[:, :bg_per_tile]
        out["bg_patches"].append(patch_cat.gather(1, keep[..., None].expand(-1, -1, patch_cat.shape[-1])).flatten(0, 1).half().cpu())
        out["thumb"].append(F.interpolate((x * std + mean).clamp(0, 1), size=32, mode="area").cpu())
    out = {k: torch.cat(v) for k, v in out.items()}
    out["slide"], out["site"] = val_ds.slide_of[indices], val_ds.site_of[indices]
    return out


emb = embed(sample_idx)
{k: tuple(v.shape) for k, v in emb.items()}

## CLS PCA + spread
Random tiles give few same-slide pairs, so `same_slide_cos` is noisy; `site_share_of_variance` vs its shuffled floor is the tissue-source-site batch effect (cf. `train.site_balance`).

In [ ]:
# PCA of the concatenated, L2-normalized CLS: spread stats, then PC1/PC2 coloured by site and by tile brightness, a thumbnail map, and the spectrum.
@torch.no_grad()
def cls_pca(emb, n_sites=8, grid=12):
    z = F.normalize(emb["cls"].flatten(1), dim=-1).to(device).double()
    zc = z - z.mean(0)
    _, S, Vh = torch.linalg.svd(zc, full_matrices=False)
    pcs = (zc @ Vh[:2].T).cpu().numpy()
    var = (S ** 2 / (S ** 2).sum()).cpu()
    cos = z @ z.T
    off = ~torch.eye(len(z), dtype=torch.bool, device=device)
    slide = torch.as_tensor(emb["slide"], device=device)
    same_slide = (slide[:, None] == slide[None]) & off

    # Share of CLS variance explained by site: between-site sum of squares over total sum of squares.
    def site_share(site):
        codes = torch.as_tensor(np.unique(site, return_inverse=True)[1], device=device)
        sums = torch.zeros(int(codes.max()) + 1, zc.shape[1], dtype=zc.dtype, device=device).index_add_(0, codes, zc)
        return ((sums ** 2).sum(1) / torch.bincount(codes)).sum().item() / zc.square().sum().item()

    stats = {
        "mean_pairwise_cos": cos[off].mean().item(),
        "mean_nn_cos": cos.masked_fill(~off, -2).amax(1).mean().item(),
        "mean_vector_norm": z.mean(0).norm().item(),  # 0 = centred on the sphere, 1 = collapsed to one direction
        "effective_rank": torch.exp(-(var * var.clamp_min(1e-12).log()).sum()).item(),
        "top10_pc_variance": var[:10].sum().item(),
        "same_slide_pairs": int(same_slide.sum()),
        "same_slide_cos": cos[same_slide].mean().item(),
        "cross_slide_cos": cos[off & ~same_slide].mean().item(),
        "site_share_of_variance": site_share(emb["site"]),
        "site_chance_share": site_share(np.random.default_rng(SEED).permutation(emb["site"])),
    }

    thumbs = emb["thumb"].permute(0, 2, 3, 1).numpy()
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.flatten()
    values, counts = np.unique(emb["site"], return_counts=True)
    axes[0].scatter(pcs[:, 0], pcs[:, 1], s=4, c="lightgray")
    for j, s in enumerate(values[np.argsort(-counts)][:n_sites]):
        m = emb["site"] == s
        axes[0].scatter(pcs[m, 0], pcs[m, 1], s=10, color=plt.cm.tab10(j), label=f"site {s} (n={m.sum()})")
    axes[0].set(title=f"CLS PC1/PC2 by tissue source site (top {n_sites})", xlabel=f"PC1 ({var[0]:.1%})", ylabel=f"PC2 ({var[1]:.1%})")
    axes[0].legend(fontsize=7)
    sc = axes[1].scatter(pcs[:, 0], pcs[:, 1], s=6, c=thumbs.mean((1, 2, 3)), cmap="viridis")
    fig.colorbar(sc, ax=axes[1], fraction=0.046, label="mean tile intensity (1 = white)")
    axes[1].set(title="CLS PC1/PC2 by tile brightness", xlabel="PC1", ylabel="PC2")
    # Thumbnail map: each occupied cell of a grid over PC1/PC2 shows the tile nearest its centre.
    lo, hi = np.quantile(pcs, 0.01, 0), np.quantile(pcs, 0.99, 0)
    step = (hi - lo) / grid
    for gx in range(grid):
        for gy in range(grid):
            d = np.linalg.norm((pcs - lo) / step - np.array([gx, gy]) - 0.5, axis=1)
            if d.min() < 0.5:
                x0, y0 = lo + np.array([gx, gy]) * step
                axes[2].imshow(thumbs[d.argmin()], extent=(x0, x0 + step[0], y0, y0 + step[1]), aspect="auto")
    axes[2].set(xlim=(lo[0], hi[0]), ylim=(lo[1], hi[1]), title="Nearest tile per PC1/PC2 grid cell", xlabel="PC1", ylabel="PC2")
    axes[3].plot(var[:256].numpy(), marker=".")
    axes[3].set(title=f"Centred CLS variance spectrum (effective rank {stats['effective_rank']:.1f})", xlabel="component", ylabel="variance fraction", yscale="log")
    plt.tight_layout()
    plt.show()
    return stats


cls_pca(emb)

## Patch tokens vs CLS

In [ ]:
# Patch-CLS cosine over all sampled tiles: distribution (concatenated and per RGB pass), the average spatial map (positional bias), and whether
# CLS is close to the mean patch token (CLS ~ patch average) or carries something patches don't, as a function of patch coherence.
@torch.no_grad()
def patch_cls(emb):
    pc = emb["patch_cls_cos"]  # (N, 4, P)
    mean_cos = F.cosine_similarity(emb["patch_mean"], F.normalize(emb["cls"].flatten(1), dim=-1), dim=-1)
    coherence = emb["patch_mean"].norm(dim=-1)  # norm of the mean unit patch: 1 = all patches identical
    grid = int(pc.shape[-1] ** 0.5)
    names = ["concat", "R pass", "G pass", "B pass"]
    stats = {f"mean_patch_cls_cos_{n}": pc[:, i].mean().item() for i, n in enumerate(names)}
    stats |= {
        "within_tile_patch_cls_std": pc[:, 0].std(1).mean().item(),
        "patch_mean_cls_cos": mean_cos.mean().item(),
        "patch_coherence": coherence.mean().item(),
    }

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for i, n in enumerate(names):
        axes[0].hist(pc[:, i].flatten().numpy(), bins=100, density=True, histtype="step", lw=1.5, label=n)
    axes[0].set(title="patch · CLS cosine (all patches)", xlabel="cos")
    axes[0].legend()
    im = axes[1].imshow(pc[:, 0].mean(0).reshape(grid, grid).numpy(), cmap="coolwarm")
    fig.colorbar(im, ax=axes[1], fraction=0.046)
    axes[1].set_title("mean patch · CLS cosine by position (concat)")
    axes[1].axis("off")
    sc = axes[2].scatter(coherence.numpy(), mean_cos.numpy(), s=6, c=emb["thumb"].mean((1, 2, 3)).numpy(), cmap="viridis")
    fig.colorbar(sc, ax=axes[2], fraction=0.046, label="mean tile intensity")
    axes[2].set(title="CLS vs mean patch token, per tile", xlabel="patch coherence", ylabel="cos(mean patch, CLS)")
    plt.tight_layout()
    plt.show()
    return stats


patch_cls(emb)

In [ ]:
# A few sampled tiles: the image, its patch · CLS cosine map, and its patches projected to RGB in a PCA basis fit on the dataset-wide patch sample.
@torch.no_grad()
def show_tiles(emb, n=4, seed=1):
    idxs = random.Random(seed).sample(sample_idx, n)
    x = torch.stack([val_ds[i]["global_views"][0] for i in idxs]).to(device)
    with torch.autocast(device.type, dtype=torch.bfloat16):
        cls, patches = model._encode(x.reshape(n * 3, 1, *x.shape[-2:]))
    cls_cat = F.normalize(cls.float().unflatten(0, (n, 3)).flatten(1), dim=-1)
    patch_cat = F.normalize(patches.float().unflatten(0, (n, 3)).transpose(1, 2).flatten(2), dim=-1)
    grid = int(patch_cat.shape[1] ** 0.5)
    cos = (patch_cat @ cls_cat[..., None])[..., 0].reshape(n, grid, grid).cpu()
    bg = emb["bg_patches"].float().to(device)
    center = bg.mean(0)
    _, _, V = torch.pca_lowrank(bg - center, q=3, center=False)
    bg_pcs = (bg - center) @ V
    lo, hi = bg_pcs.quantile(0.01, dim=0), bg_pcs.quantile(0.99, dim=0)
    rgb = (((patch_cat - center) @ V - lo) / (hi - lo)).clamp(0, 1).reshape(n, grid, grid, 3).cpu()
    img = (x.cpu() * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1).permute(0, 2, 3, 1)

    extent = (0, img.shape[2], img.shape[1], 0)
    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
    for r in range(n):
        axes[r, 0].imshow(img[r])
        axes[r, 0].set_title(f"val tile {idxs[r]}")
        axes[r, 1].imshow(img[r])
        im = axes[r, 1].imshow(cos[r], cmap="coolwarm", alpha=0.6, extent=extent, interpolation="nearest")
        fig.colorbar(im, ax=axes[r, 1], fraction=0.046)
        axes[r, 1].set_title("patch · CLS cosine")
        axes[r, 2].imshow(rgb[r], interpolation="nearest")
        axes[r, 2].set_title("patch PCA → RGB (dataset basis)")
        for ax in axes[r]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()


show_tiles(emb)

## RGB-pass redundancy
GenBio runs each RGB plane through the backbone separately and the probe concatenates the three CLS tokens (4608 dims). Like our factored-head chunks, the question is whether the parts encode different things.

In [ ]:
# Per-tile cosine between pass CLS tokens (same weights, so one shared space) and linear CKA across tiles (1 = identical geometry up to rotation/scale).
@torch.no_grad()
def pass_redundancy(emb):
    cls = F.normalize(emb["cls"], dim=-1).to(device).double()  # (N, 3, D)
    stats = {}
    plt.figure(figsize=(6, 4))
    for a, b in [(0, 1), (0, 2), (1, 2)]:
        cos = (cls[:, a] * cls[:, b]).sum(-1)
        xa, xb = cls[:, a] - cls[:, a].mean(0), cls[:, b] - cls[:, b].mean(0)
        cka = (xa.T @ xb).square().sum() / ((xa.T @ xa).square().sum() * (xb.T @ xb).square().sum()).sqrt()
        name = "RGB"[a] + "RGB"[b]
        stats[name] = {"tile_cos": cos.mean().item(), "linear_cka": cka.item()}
        plt.hist(cos.cpu().numpy(), bins=50, alpha=0.5, label=name)
    plt.gca().set(title="same-tile CLS cosine between RGB passes", xlabel="cos")
    plt.legend()
    plt.show()
    return stats


pass_redundancy(emb)